# 03 · SAR ground-truth loop — did water actually pool where we predicted?

Sentinel-1 radar sees through monsoon clouds. Smooth standing water reflects the radar pulse
away from the satellite, so flooded pixels go *dark* (low VV backscatter). This notebook:
1. builds a dry-season reference image,
2. maps observed standing water after any monsoon date you pick,
3. scores notebook 01's predictions with proper verification metrics (POD / FAR / CSI),
4. loops over several monsoon seasons to build an **observed waterlogging frequency map** —
   arguably the most valuable single product of this whole project.

**Requires:** notebook 01 outputs in `WORK`.

In [ ]:
import sys, subprocess, os

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install("rasterio")

WORK = "/content/drive/MyDrive/floodtwin" if os.path.isdir("/content/drive/MyDrive") else "/content/floodtwin"
assert os.path.exists(f"{WORK}/depth.tif"), "Run notebook 01 first."

import ee
PROJECT_ID = "your-cloud-project-id"   # <-- EDIT
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT_ID)

AOI = (85.02, 25.54, 85.30, 25.68)
region = ee.Geometry.Rectangle(list(AOI))
SCALE = 30

In [ ]:
import requests

def download_ee_image(img, path, scale=SCALE, bands=None):
    if bands is not None:
        img = img.select(bands)
    url = img.getDownloadURL({"region": region, "scale": scale,
                              "format": "GEO_TIFF", "crs": "EPSG:4326"})
    r = requests.get(url, timeout=600); r.raise_for_status()
    open(path, "wb").write(r.content); print("saved", path)

def s1(start, end):
    """Sentinel-1 IW VV collection over the AOI, speckle-filtered, in dB."""
    col = (ee.ImageCollection("COPERNICUS/S1_GRD")
           .filterBounds(region).filterDate(start, end)
           .filter(ee.Filter.eq("instrumentMode", "IW"))
           .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
           .select("VV"))
    return col.map(lambda im: im.focal_median(50, "circle", "meters")
                                .copyProperties(im, ["system:time_start"]))

# Dry-season reference: median of Feb-Mar passes (edit the year range as you like).
dry_ref = s1("2025-02-01", "2025-03-31").median()

# List available monsoon acquisition dates so you can pick one just after a big rain.
monsoon = s1("2025-06-01", "2025-10-15")
dates = monsoon.aggregate_array("system:time_start").getInfo()
import datetime
print("Sentinel-1 passes in monsoon 2025:")
for t in dates:
    print(" ", datetime.datetime.utcfromtimestamp(t / 1000).strftime("%Y-%m-%d"))

## Observed water mask for one event
Two complementary tests, OR-combined: an absolute threshold (VV below −16 dB ≈ open water)
and a change test (≥4 dB darker than the dry reference, which catches shallower urban
waterlogging). Permanent water (Ganga, ponds) is masked out with JRC so we score only
*flood* water.

In [ ]:
EVENT_DATE = "2025-07-15"   # <-- EDIT: pick a date from the list above, ideally right after heavy rain
import datetime as _dt
d0 = _dt.date.fromisoformat(EVENT_DATE)
event_img = s1(str(d0 - _dt.timedelta(days=1)), str(d0 + _dt.timedelta(days=2))).mosaic()

jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").unmask(0)
water = (event_img.lt(-16).Or(event_img.subtract(dry_ref).lt(-4))
         .updateMask(jrc.lt(50)).unmask(0)
         .focal_mode(60, "circle", "meters"))           # despeckle the binary mask
download_ee_image(water, f"{WORK}/observed_water_{EVENT_DATE}.tif")

## Score the prediction
Standard forecast-verification metrics over land pixels:
**POD** (fraction of observed water we predicted), **FAR** (fraction of our predictions that
were dry), **CSI** (overall skill, 0–1).

In [ ]:
import numpy as np, rasterio

def read1(p):
    with rasterio.open(p) as s:
        return s.read(1)

obs = read1(f"{WORK}/observed_water_{EVENT_DATE}.tif")
pred_depth = read1(f"{WORK}/depth.tif")
r, c = min(obs.shape[0], pred_depth.shape[0]), min(obs.shape[1], pred_depth.shape[1])
obs = obs[:r, :c] > 0.5
pred = pred_depth[:r, :c] > 0.15

hits = int((obs & pred).sum()); misses = int((obs & ~pred).sum()); fa = int((~obs & pred).sum())
pod = hits / max(hits + misses, 1)
far = fa / max(hits + fa, 1)
csi = hits / max(hits + misses + fa, 1)
print(f"POD={pod:.2f}  FAR={far:.2f}  CSI={csi:.2f}   (hits {hits}, misses {misses}, false alarms {fa})")
print("Interpretation: CSI > 0.3 on 30 m urban data is already respectable;")
print("misses cluster where the DEM is wrong or sewers exist — those pixels are gold for DEM correction.")

## Waterlogging frequency climatology
Loop over every monsoon pass for several years and count how often each pixel is wet.
This 'how often does it actually flood here' map is independent of any model — pure observation —
and is the calibration target for notebooks 04 and 05.

In [ ]:
YEARS = [2023, 2024, 2025]
freq = ee.Image(0).rename("freq"); npass = 0
for y in YEARS:
    col = s1(f"{y}-06-01", f"{y}-10-15")
    nimg = col.size().getInfo()
    if nimg == 0:
        continue
    wet = col.map(lambda im: im.lt(-16).Or(im.subtract(dry_ref).lt(-4)).unmask(0))
    freq = freq.add(wet.sum()); npass += nimg
freq = freq.divide(max(npass, 1)).updateMask(jrc.lt(50))
download_ee_image(freq, f"{WORK}/waterlogging_frequency.tif")
print(f"frequency map built from {npass} radar passes")

In [ ]:
import folium
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from PIL import Image

fr = read1(f"{WORK}/waterlogging_frequency.tif")[:r, :c]
fr = np.nan_to_num(fr)
rgba = cm.plasma(mcolors.Normalize(0, max(0.3, fr.max()))(fr))
rgba[..., 3] = np.where(fr > 0.05, 0.8, 0.0)
Image.fromarray((rgba * 255).astype(np.uint8)).save(f"{WORK}/freq_overlay.png")

with rasterio.open(f"{WORK}/depth.tif") as s:
    T = s.transform
lon0, lat1 = T * (0, 0); lon1, lat0 = T * (c, r)
m = folium.Map(location=[(lat0 + lat1) / 2, (lon0 + lon1) / 2], zoom_start=12, tiles="cartodbpositron")
folium.raster_layers.ImageOverlay(f"{WORK}/freq_overlay.png",
                                  bounds=[[lat0, lon0], [lat1, lon1]]).add_to(m)
m.save(f"{WORK}/frequency_map.html")
m

## Caveats and the feedback loop
- C-band radar struggles in dense built-up cores: tall buildings cause layover/shadow and
  double-bounce can make flooded streets *bright* instead of dark. Expect the cleanest signal
  in open areas, parks, fields and wide roads; treat dense-core pixels with suspicion.
- 30 m pixels miss narrow street flooding. The frequency map is a lower bound.
- **Feedback loop:** pixels where observation and prediction persistently disagree are exactly
  where the DEM is lying. Export them (`obs != pred`) as training targets and you have the
  'complaint-corrected DEM' idea almost for free — radar complaints instead of human ones.